In [12]:
import numpy as np

In [13]:
class GridWorld:
    def __init__(self, grid_size, n_holes=1, goal_state=None):
        self.rows, self.cols = grid_size
        
        self.goal_state = (self.rows-1, self.cols-1) if goal_state is None else goal_state   # bottom right
        self.n_holes = n_holes
        self.hole_states = self.set_holes()

        self.n_states = self.rows * self.cols
        self.action_space = [0, 1, 2, 3]  # up, down, left, right

        self.agent_pos = None

    def set_holes(self):
        n_holes = 0
        hole_states = []

        while n_holes < self.n_holes:
            while True:
                hole = (np.random.randint(0, self.rows-1), np.random.randint(0, self.cols-1))
                if hole != self.goal_state and hole not in hole_states:
                    n_holes += 1
                    hole_states.append(hole)
                    break
        return hole_states
                    

    def reset(self):
        while True:
            row = np.random.randint(0, self.rows)
            col = np.random.randint(0, self.cols)

            if (row, col) != self.goal_state:   # add any other terminal states so episode doesnt end immediately
                break
        
        self.agent_pos = (row, col)
        return self.agent_pos

    def step(self, action):

        # returns: agent pos, reward, terminated?

        row, col = self.agent_pos

        # ensure to stay within grid bounds
        if action == 0:
            row = max(row-1, 0)
        elif action == 1:
            row = min(row+1, self.rows-1)
        elif action == 2:
            col = max(col-1, 0)
        elif action == 3:
            col = min(col+1, self.cols-1)

        self.agent_pos = (row, col)

        if self.agent_pos == self.goal_state:
            return self.agent_pos, 10, True
        
        elif self.agent_pos in self.hole_states:
            return self.agent_pos, -5, False
        else:
            return self.agent_pos, -1, False



In [ ]:
class TemporalDifferenceAgent:
    def __init__(self, action_space, alpha=0.1, gamma=0.9, method=None, n=1, epsilon=0.5, epsilon_decay=0.999, min_epsilon=0.0):
        self.action_space = action_space
        self.alpha = alpha
        self.gamma = gamma
        self.method = method    # SARSA or Q-learning
        self.n = n              # n step TD
        self.epsilon = epsilon
        self.epsilon_decay = epsilon_decay
        self.min_epsilon = min_epsilon

        # Q(s,a)
        self.q_table = {}   # for each state, store values of each of 4 actions

    def get_q_values(self, state):

        if state not in self.q_table:
            self.q_table[state] = np.zeros(len(self.action_space))  # init each new state with 0 for each action

        return self.q_table[state]  # array of 4 actions
    
    def get_action(self, state):

        # epsilon greedy policy

        # 1. Explore
        if np.random.uniform(0, 1) < self.epsilon:
            return np.random.choice(self.action_space)
        
        # 2. Exploit
        q_values = self.get_q_values(state)
        return np.argmax(q_values)
    

    def update(self, memory):

        # Q(s, a) = 𝔼[ R₁ + γR₂ + γ²R₃ + … + γⁿ⁻¹Rₙ + γⁿV(sₙ) ]

        # SARSA needs all 5
        # Q-Learning doesnt need next action

        states, actions, R, next_states, next_actions, done = zip(*memory)

        # 1. Compute Return: G = R₁ + γR₂ + γ²R₃ + … + γⁿ⁻¹Rₙ + γⁿV(sₙ)
        G = sum(self.gamma**i * R[i] for i in range(len(memory)))

        q_values = self.get_q_values(states[0]) # all q_vals for state
        old_q = q_values[actions[0]]    # Q(s,a)
        
        # bootstrap of last next state if not terminal
        if True not in done:    
            next_q_values = self.get_q_values(next_states[-1])

            if self.method == "sarsa":
                new_q = next_q_values[next_actions[-1]]  # Q(s',a')       bootstrap off last state's next action

            elif self.method == "qlearning":
                new_q = np.max(next_q_values)    # max_a Q(s',a)     bootstrap off last state's best action

            G += self.gamma**len(memory) * new_q    # G += γⁿV(sₙ)

        # if episode finishes in sequence then final next_state must have 0 reward
        # so G has no bootstrapping

        # Q(s,a) = Q(s,a) + alpha * (G - Q(s,a))
        self.q_table[states[0]][actions[0]] = old_q + self.alpha * (G - old_q)



In [15]:
# MAIN TRAINING LOOP
from collections import deque


env = GridWorld(grid_size=(19,19), n_holes=19, goal_state=(4,4))
agent = TemporalDifferenceAgent(env.action_space, method="qlearning") #sarsa

episodes = 5000
incomplete_episodes = 0


for e in range(episodes):
    state = env.reset()
    action = agent.get_action(state)

    done = False    # track if episode finishes (terminal state reached)
    steps = 0

    memory = deque()     # for n steps TD

    while not done:
        next_state, reward, done = env.step(action)
        next_action = agent.get_action(next_state)

        memory.append((state, action, reward, next_state, next_action, done))

        # learn every n steps
        if len(memory) >= agent.n:
            agent.update(memory)  
            memory.popleft()    # remove first SARSA tuple    

        state = next_state
        action = next_action

        steps += 1

        if steps > 500:    # prevent infinite loop - never reaches terminal
            incomplete_episodes += 1
            break
    
    # drain final states that dont have full n-steps
    while len(memory) > 0:
        agent.update(memory)  
        memory.popleft() 

    # decay eps after each episode to reduce exploration over time
    agent.epsilon = max(agent.min_epsilon, agent.epsilon * agent.epsilon_decay)

    # prints
    if e % 500 == 0:
        print(f"Episode: {e}   Current Epsilon: {agent.epsilon}   Steps: {steps}")


print(f"\nIncomplete Episodes: {incomplete_episodes}")

Episode: 0   Current Epsilon: 0.4995   Steps: 501
Episode: 500   Current Epsilon: 0.30288628295816183   Steps: 74
Episode: 1000   Current Epsilon: 0.18366386467309628   Steps: 21
Episode: 1500   Current Epsilon: 0.11136990046959969   Steps: 9
Episode: 2000   Current Epsilon: 0.06753236273605094   Steps: 8
Episode: 2500   Current Epsilon: 0.04095020285986938   Steps: 10
Episode: 3000   Current Epsilon: 0.024831340802019108   Steps: 15
Episode: 3500   Current Epsilon: 0.015057202235016836   Steps: 15
Episode: 4000   Current Epsilon: 0.009130370403830978   Steps: 13
Episode: 4500   Current Epsilon: 0.005536464371666822   Steps: 8

Incomplete Episodes: 28


In [16]:
# Visualisation ######## goal state never reached????? in qtable

# up down inverted since (0,0) is top left
actions = [ "↑", "↓", "←", "→"]

for row in range(env.rows):
    for col in range(env.cols):

        state = (row, col)

        if state == env.goal_state:
            print("G", end=" ")
        elif state in env.hole_states:
            print("H", end=" ")
        else:
            optimal_action = np.argmax(agent.q_table[state]) # assumes every state has been visited during learning
            print(actions[optimal_action], end=" ")
    print()



→ → → → ↓ ← ↓ ↓ ← ↓ ← ↓ ↓ ← ↓ ← → ↓ ← 
↓ H ↓ ↓ ↓ ← ↓ ↓ ← ↓ ← ↓ ↓ ← ↓ ↓ ← ↓ ↓ 
H → → → ↓ ← ↓ ↓ ← ↓ ← ↓ ← ← ↓ ← ← ← ↓ 
→ ↓ ↓ → ↓ ← ↓ ↓ ← ↓ ↓ ↓ ← H ↓ ← ↓ ↓ ← 
H → → → G ← ← ← ← ← ← ← ← ← ← ← ← ← ← 
→ → → → ↑ ← ← ← ← ← ← ← ← ← ← ← ← ← ← 
→ → → ↑ ↑ ← ← ← ↑ ← ↑ ↑ ↑ ↑ ← ← ← ← ← 
→ → ↑ ↑ ↑ ↑ ↑ ↑ H ↑ ↑ ↑ ↑ H ↑ ← ↑ H ↑ 
↑ ↑ ↑ ↑ ↑ ↑ ← H ↓ ↑ ↑ ↑ ↑ ← ↑ ← ← H ↑ 
↑ ↑ ↑ → ↑ H ↑ ← ← ← ← H ↑ H ↑ ↑ ↑ ↓ ← 
→ → → → ↑ ← ↑ ↑ ← ← ← ← ← ← ↑ ← ← ↑ ↑ 
→ → → → ↑ ← ↑ H ↑ ↑ ↑ ↑ ← ← ↑ ↑ ← ← ↑ 
→ → → ↑ ↑ ↑ H → ↑ ↑ ← ← ← ← ↑ ↑ → ← ↓ 
→ → → ↑ ↑ ↑ ← ← ↑ ↑ ← ↑ ↑ ← ↑ ← ↑ H ↑ 
↑ → → ↑ H ↑ ← ← ↑ ↑ ↑ H ↑ H ↑ ↑ ↑ ← ↑ 
↓ → → ↑ ← ↑ ← ← ← ↑ ← ← ← ↓ ← ← ↑ ← ↓ 
→ ↑ → ↑ ↑ ↑ ← ← ↑ ← ↑ ← ← ↑ ← ← ← ↓ ← 
← → → ↑ ↓ ↑ ↑ ← ↑ ← ← → ← H ↓ ← ↑ ↑ ← 
← ↑ ↓ ↑ ← ↑ ← ← ↓ ← → ← ↓ → ↑ ← → → ↑ 
